In [1]:
"""
使用 DuckDB 和 Polars 分析 large_data.csv 中 device_info 字段的操作系统分布
- 从 JSON 字符串中提取 os 字段
- 按不同操作系统统计请求数及占比
- 处理 JSON 解析失败的情况并标注为 unknown
"""

import duckdb
import polars as pl

CSV_FILE = r"D:\MyProjects\DataAnylyse\lab01\large_data.csv"

# =============================================================================
# 方法 1: 使用 DuckDB 直接查询 CSV（流式处理，不加载全量数据到内存）
# =============================================================================
print("=" * 60)
print("【DuckDB 分析】")
print("=" * 60)

# DuckDB 直接查询 CSV 文件，使用 json_extract_string 解析 JSON
# TRY 开头的函数在解析失败时返回 NULL 而非报错
duckdb_query = """
WITH parsed AS (
    SELECT 
        COALESCE(
            TRY(json_extract_string(device_info::VARCHAR, '$.os')), 
            'unknown'
        ) as os
    FROM read_csv_auto('{csv_file}')
)
SELECT 
    os,
    COUNT(*) as count
FROM parsed
GROUP BY os
ORDER BY count DESC
""".format(csv_file=CSV_FILE)

duckdb_result = duckdb.query(duckdb_query).df()

total_count = duckdb_result['count'].sum()
duckdb_result['ratio'] = (duckdb_result['count'] / total_count * 100).round(4)

print(f"\n数据文件：{CSV_FILE}")
print(f"总记录数：{total_count:,}\n")

print("操作系统分布统计:")
print("-" * 50)
print(f"{'操作系统':<20} {'记录数':>12} {'占比':>12}")
print("-" * 50)
for _, row in duckdb_result.iterrows():
    os_name = row['os'] if row['os'] else 'unknown'
    print(f"{os_name:<20} {int(row['count']):>12,} {row['ratio']:>11.4f}%")
print("-" * 50)


# =============================================================================
# 方法 2: 使用 Polars 进行相同分析
# =============================================================================
print("\n" + "=" * 60)
print("【Polars 分析】")
print("=" * 60)

# Polars 流式读取 CSV
df = pl.scan_csv(CSV_FILE)

# 使用正则表达式从 JSON 字符串中提取 os 字段值
# 解析失败时返回 null，用 fill_null 填充为 'unknown'
df_parsed = df.with_columns(
    pl.col('device_info')
    .str.extract(r'"os"\s*:\s*"([^"]+)"', 1)
    .fill_null('unknown')
    .alias('os')
)

# 统计各操作系统的数量
os_stats = (
    df_parsed
    .group_by('os')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .collect()
)

total_count_pl = os_stats['count'].sum()
os_stats = os_stats.with_columns(
    (pl.col('count') / total_count_pl * 100).alias('ratio')
)

print(f"\n数据文件：{CSV_FILE}")
print(f"总记录数：{total_count_pl:,}\n")

print("操作系统分布统计:")
print("-" * 50)
print(f"{'操作系统':<20} {'记录数':>12} {'占比':>12}")
print("-" * 50)
for row in os_stats.iter_rows():
    os_name = row[0] if row[0] else 'unknown'
    print(f"{os_name:<20} {int(row[1]):>12,} {row[2]:>11.4f}%")
print("-" * 50)


# =============================================================================
# 总结
# =============================================================================
print("\n" + "=" * 60)
print("【分析总结】")
print("=" * 60)

print(f"""
数据文件：{CSV_FILE}
总记录数：{total_count:,}

操作系统分布 (共 {len(duckdb_result)} 种):
""")

for _, row in duckdb_result.iterrows():
    os_name = row['os'] if row['os'] else 'unknown'
    marker = " ← 解析失败" if os_name == 'unknown' else ""
    print(f"  - {os_name}: {int(row['count']):,} 条 ({row['ratio']:.4f}%){marker}")

unknown_count = duckdb_result[duckdb_result['os'] == 'unknown']['count'].sum() if 'unknown' in duckdb_result['os'].values else 0
print(f"""
JSON 解析失败记录数：{unknown_count:,} 条 ({unknown_count/total_count*100:.4f}%)
""")

【DuckDB 分析】

数据文件：D:\MyProjects\DataAnylyse\lab01\large_data.csv
总记录数：1,000,000

操作系统分布统计:
--------------------------------------------------
操作系统                          记录数           占比
--------------------------------------------------
macOS                     200,403     20.0403%
Windows                   200,054     20.0054%
iOS                       200,054     20.0054%
Linux                     199,828     19.9828%
Android                   199,661     19.9661%
--------------------------------------------------

【Polars 分析】

数据文件：D:\MyProjects\DataAnylyse\lab01\large_data.csv
总记录数：1,000,000

操作系统分布统计:
--------------------------------------------------
操作系统                          记录数           占比
--------------------------------------------------
macOS                     200,403     20.0403%
Windows                   200,054     20.0054%
iOS                       200,054     20.0054%
Linux                     199,828     19.9828%
Android                   199,661     19.9661

In [2]:
"""
使用 DuckDB 和 Polars 分析 large_data.csv 中 user_id 异常数据
- 统计 user_id = -1 的记录数
- 统计 user_id = 'guest' 的记录数
- 计算各自占总记录数的比例
"""

import duckdb
import polars as pl

CSV_FILE = r"D:\MyProjects\DataAnylyse\lab01\large_data.csv"

# =============================================================================
# 方法 1: 使用 DuckDB 直接查询 CSV（流式处理，不加载全量数据到内存）
# =============================================================================
print("=" * 60)
print("【DuckDB 分析】")
print("=" * 60)

# DuckDB 直接查询 CSV 文件
duckdb_query = """
SELECT 
    COUNT(*) as total_count,
    SUM(CASE WHEN user_id = '-1' THEN 1 ELSE 0 END) as user_id_neg1_count,
    SUM(CASE WHEN user_id = 'guest' THEN 1 ELSE 0 END) as user_id_guest_count
FROM read_csv_auto('{csv_file}')
""".format(csv_file=CSV_FILE)

result = duckdb.query(duckdb_query).df().iloc[0]

total_count = int(result['total_count'])
neg1_count = int(result['user_id_neg1_count'])
guest_count = int(result['user_id_guest_count'])

neg1_ratio = neg1_count / total_count * 100 if total_count > 0 else 0
guest_ratio = guest_count / total_count * 100 if total_count > 0 else 0

print(f"""
数据文件：{CSV_FILE}

【异常类型统计】
┌─────────────────────┬──────────────┬──────────────┐
│ 异常类型            │ 记录数       │ 占比         │
├─────────────────────┼──────────────┼──────────────┤
│ user_id = -1        │ {neg1_count:>10,} │ {neg1_ratio:>10.4f}% │
│ user_id = 'guest'   │ {guest_count:>10,} │ {guest_ratio:>10.4f}% │
├─────────────────────┼──────────────┼──────────────┤
│ 异常合计            │ {neg1_count + guest_count:>10,} │ {(neg1_ratio + guest_ratio):>10.4f}% │
├─────────────────────┼──────────────┼──────────────┤
│ 总记录数            │ {total_count:>10,} │ 100.0000%    │
└─────────────────────┴──────────────┴──────────────┘
""")


# =============================================================================
# 方法 2: 使用 Polars 进行相同分析
# =============================================================================
print("=" * 60)
print("【Polars 分析】")
print("=" * 60)

# Polars 流式读取 CSV
df = pl.scan_csv(CSV_FILE)

# 统计总数
total_count_pl = df.select(pl.len()).collect().item()

# 统计 user_id = '-1' 的数量 (先 collect 再过滤，避免 lazy 执行计划问题)
df_collected = df.collect()
neg1_count_pl = df_collected.filter(pl.col('user_id') == '-1').shape[0]

# 统计 user_id = 'guest' 的数量
guest_count_pl = df_collected.filter(pl.col('user_id') == 'guest').shape[0]

neg1_ratio_pl = neg1_count_pl / total_count_pl * 100 if total_count_pl > 0 else 0
guest_ratio_pl = guest_count_pl / total_count_pl * 100 if total_count_pl > 0 else 0

print(f"""
数据文件：{CSV_FILE}

【异常类型统计】
┌─────────────────────┬──────────────┬──────────────┐
│ 异常类型            │ 记录数       │ 占比         │
├─────────────────────┼──────────────┼──────────────┤
│ user_id = -1        │ {neg1_count_pl:>10,} │ {neg1_ratio_pl:>10.4f}% │
│ user_id = 'guest'   │ {guest_count_pl:>10,} │ {guest_ratio_pl:>10.4f}% │
├─────────────────────┼──────────────┼──────────────┤
│ 异常合计            │ {neg1_count_pl + guest_count_pl:>10,} │ {(neg1_ratio_pl + guest_ratio_pl):>10.4f}% │
├─────────────────────┼──────────────┼──────────────┤
│ 总记录数            │ {total_count_pl:>10,} │ 100.0000%    │
└─────────────────────┴──────────────┴──────────────┘
""")


# =============================================================================
# 总结
# =============================================================================
print("=" * 60)
print("【分析总结】")
print("=" * 60)

anomaly_total = neg1_count + guest_count
anomaly_ratio = neg1_ratio + guest_ratio

print(f"""
异常数据汇总:
  - user_id = -1  的记录数：{neg1_count:,} 条 (占比 {neg1_ratio:.4f}%)
  - user_id = 'guest' 的记录数：{guest_count:,} 条 (占比 {guest_ratio:.4f}%)
  
  异常数据总计：{anomaly_total:,} 条 (占比 {anomaly_ratio:.4f}%)
  正常数据总计：{total_count - anomaly_total:,} 条 (占比 {100 - anomaly_ratio:.4f}%)
""")

【DuckDB 分析】

数据文件：D:\MyProjects\DataAnylyse\lab01\large_data.csv

【异常类型统计】
┌─────────────────────┬──────────────┬──────────────┐
│ 异常类型            │ 记录数       │ 占比         │
├─────────────────────┼──────────────┼──────────────┤
│ user_id = -1        │      1,986 │     0.1986% │
│ user_id = 'guest'   │      1,998 │     0.1998% │
├─────────────────────┼──────────────┼──────────────┤
│ 异常合计            │      3,984 │     0.3984% │
├─────────────────────┼──────────────┼──────────────┤
│ 总记录数            │  1,000,000 │ 100.0000%    │
└─────────────────────┴──────────────┴──────────────┘

【Polars 分析】

数据文件：D:\MyProjects\DataAnylyse\lab01\large_data.csv

【异常类型统计】
┌─────────────────────┬──────────────┬──────────────┐
│ 异常类型            │ 记录数       │ 占比         │
├─────────────────────┼──────────────┼──────────────┤
│ user_id = -1        │      1,986 │     0.1986% │
│ user_id = 'guest'   │      1,998 │     0.1998% │
├─────────────────────┼──────────────┼──────────────┤
│ 异常合计            │      3,9

In [3]:
"""
使用 DuckDB 和 Polars 分析 large_data.csv 中的 event_type 字段
- DuckDB: 直接对本地 CSV 发起查询，无需加载全量内存
- Polars: 高效数据处理和统计
"""

import duckdb
import polars as pl

CSV_FILE = "large_data.csv"

# =============================================================================
# 方法 1: 使用 DuckDB 直接查询 CSV（流式处理，不加载全量数据到内存）
# =============================================================================
print("=" * 60)
print("【DuckDB 分析】")
print("=" * 60)

# DuckDB 直接查询 CSV 文件
duckdb_query = """
SELECT 
    event_type,
    COUNT(*) as count
FROM read_csv_auto('{csv_file}')
GROUP BY event_type
ORDER BY count DESC
""".format(csv_file=CSV_FILE)

duckdb_result = duckdb.query(duckdb_query).df()
print("\n所有唯一 event_type 及其出现次数:")
print(duckdb_result.to_string(index=False))

# 计算总记录数
total_count = duckdb_result['count'].sum()
print(f"\n总记录数：{total_count:,}")

# 定义正常的事件类型（用于识别异常）
normal_event_types = {'click', 'login', 'logout', 'payment'}

# 找出异常数据（拼写错误）
duckdb_result['is_anomaly'] = ~duckdb_result['event_type'].isin(normal_event_types)
anomalies = duckdb_result[duckdb_result['is_anomaly']]
normal_data = duckdb_result[~duckdb_result['is_anomaly']]

anomaly_count = anomalies['count'].sum()
anomaly_ratio = anomaly_count / total_count * 100

print(f"\n正常 event_type ({len(normal_data)} 种):")
for _, row in normal_data.iterrows():
    print(f"  - {row['event_type']}: {row['count']:,} ({row['count']/total_count*100:.2f}%)")

print(f"\n异常 event_type ({len(anomalies)} 种，疑似拼写错误):")
for _, row in anomalies.iterrows():
    print(f"  - {row['event_type']}: {row['count']:,} ({row['count']/total_count*100:.4f}%)")

print(f"\n异常数据总量：{anomaly_count:,} 条")
print(f"异常数据占比：{anomaly_ratio:.4f}%")


# =============================================================================
# 方法 2: 使用 Polars 进行相同分析
# =============================================================================
print("\n" + "=" * 60)
print("【Polars 分析】")
print("=" * 60)

# Polars 流式读取 CSV
df = pl.scan_csv(CSV_FILE)

# 统计 event_type 分布
event_type_stats = (
    df
    .group_by('event_type')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .collect()
)

print("\n所有唯一 event_type 及其出现次数:")
print(str(event_type_stats))

total_count_pl = event_type_stats['count'].sum()
print(f"\n总记录数：{total_count_pl:,}")

# 识别异常数据
event_type_stats = event_type_stats.with_columns(
    pl.col('event_type').is_in(normal_event_types).alias('is_normal')
)

anomalies_pl = event_type_stats.filter(pl.col('is_normal') == False)
normal_data_pl = event_type_stats.filter(pl.col('is_normal') == True)

anomaly_count_pl = anomalies_pl['count'].sum()
anomaly_ratio_pl = anomaly_count_pl / total_count_pl * 100

print(f"\n正常 event_type ({len(normal_data_pl)} 种):")
for row in normal_data_pl.iter_rows():
    print(f"  - {row[0]}: {row[1]:,} ({row[1]/total_count_pl*100:.2f}%)")

print(f"\n异常 event_type ({len(anomalies_pl)} 种，疑似拼写错误):")
for row in anomalies_pl.iter_rows():
    print(f"  - {row[0]}: {row[1]:,} ({row[1]/total_count_pl*100:.4f}%)")

print(f"\n异常数据总量：{anomaly_count_pl:,} 条")
print(f"异常数据占比：{anomaly_ratio_pl:.4f}%")

# =============================================================================
# 总结
# =============================================================================
print("\n" + "=" * 60)
print("【分析总结】")
print("=" * 60)
print(f"""
数据文件：{CSV_FILE}
总记录数：{total_count:,}

正常事件类型 (4 种): click, login, logout, payment
异常事件类型 ({len(anomalies)} 种): {', '.join(anomalies['event_type'].tolist())}

异常数据统计:
  - 异常记录数：{anomaly_count:,}
  - 异常占比：{anomaly_ratio:.4f}%
  - 正常占比：{100 - anomaly_ratio:.4f}%
""")

【DuckDB 分析】

所有唯一 event_type 及其出现次数:
event_type  count
     login 245887
    logout 244870
   payment 244592
     click 244536
     lgoin   1316
      logn   1313
      pymt   1306
     lgout   1292
      clic   1291
      clik   1279
     logot   1253
      logi   1248
    payent   1247
   paymant   1242
     loout   1238
     clcik   1233
     logut   1232
     clikc   1227
     loign   1226
    paymet   1172

总记录数：1,000,000

正常 event_type (4 种):
  - login: 245,887 (24.59%)
  - logout: 244,870 (24.49%)
  - payment: 244,592 (24.46%)
  - click: 244,536 (24.45%)

异常 event_type (16 种，疑似拼写错误):
  - lgoin: 1,316 (0.1316%)
  - logn: 1,313 (0.1313%)
  - pymt: 1,306 (0.1306%)
  - lgout: 1,292 (0.1292%)
  - clic: 1,291 (0.1291%)
  - clik: 1,279 (0.1279%)
  - logot: 1,253 (0.1253%)
  - logi: 1,248 (0.1248%)
  - payent: 1,247 (0.1247%)
  - paymant: 1,242 (0.1242%)
  - loout: 1,238 (0.1238%)
  - clcik: 1,233 (0.1233%)
  - logut: 1,232 (0.1232%)
  - clikc: 1,227 (0.1227%)
  - loign: 1,226 (0.1226%)